In [ ]:
import numpy as np 
from scipy import sparse 
from matplotlib import pyplot as plt
from matplotlib import animation
from time import time
from IPython.display import HTML
from matplotlib.animation import FuncAnimation
from tqdm import tqdm
import scipy


In [23]:
def exponential_heat_solver_2d_boundary(nu, u0, x_span, y_span, tf, nx, ny, nt, neumann_start = False, nuemann_end = False, boundary = [0,0]):
    # define space and time domains
    x_vals, dx = np.linspace(*x_span, nx+1, retstep=True)
    y_vals, dy = np.linspace(*y_span, ny+1, retstep=True)
    t_vals, dt = np.linspace(0, tf, nt+1, retstep=True)
    X, Y = np.meshgrid(x_vals, y_vals)

    # pre-compute these values
    dx2_inv = 1. / dx**2
    dy2_inv = 1. / dy**2

    # compute the initial state
    U_init = np.array(
        [[u0(X[i, j], Y[i, j]) for i in range(X.shape[0])] for j in range(X.shape[1])]
    )

    # define differential operator
    T = sparse.diags(
        diagonals=[dx2_inv, -2.*(dx2_inv+dy2_inv), dx2_inv],
        offsets=[-1, 0, 1],
        shape=(nx+1, nx+1)
    )
    T = sparse.csr_matrix(T)
    C = np.zeros(nx+1)
    if neumann_start:
        T[0,1] +=  dx2_inv
        C[0] -= 2 * boundary[0] / dx
    else:
        T[0] = 0
        T[0,0] = 1
    
    if nuemann_end:
        T[-1,-2] += dx2_inv
        C[-1] -= 2 * boundary[-1] / dx
    else:
        T[-1] = 0
        T[-1,-1] = 1
        
    
    L = sparse.block_diag([T] * (ny+1))
    L.setdiag(dy2_inv, -(ny+1))
    L.setdiag(dy2_inv, ny+1)
    # L = L.toarray()
    L = L.tocsr()
    
    C = np.array([C] * (ny+1)).flatten()

    # exponentiate the linear operator via diagonalization
    exp_operator = sparse.linalg.expm(dt * nu * L)
    
    I = sparse.eye(L.shape[0])
    phi = sparse.linalg.spsolve(nu * L, (exp_operator - I) @ C)
    

    # initialize solution matrix -- homogenous Dirichlet BCs
    U = np.zeros((nt+1, nx+1, ny+1))
    sl = np.s_[1:-1]                    # interior slice 
    if not neumann_start:
        U[0,:,0] = boundary[0]
    if not nuemann_end:
        U[0,:,-1] = boundary[-1]
    U[0, sl, sl] = U_init[sl, sl]       # set initial state on interior

    start = time()
    # solver loop
    for t in tqdm(range(1, nt+1)):
        U[t] = (exp_operator @ U[t-1].flatten() + phi).reshape((nx+1, ny+1))

    return U, X, Y, t_vals, time() - start

In [24]:
# model parameters 
nu = 1.             # diffusion coefficient
x0, xf = 0, 1       # space domain (x)
y0, yf = 0, 1       # space domain (y)
tf = 0.25           # time domain
def u0(x, y):       # initial state
    return (
        np.sin(np.pi * (x - x0) / (xf - x0)) * 
        np.sin(np.pi * (y - y0) / (yf - y0))
    )

# system parameters 
nx = 50 
ny = 50
nt = 100 


U_edt, X, Y, t_vals, end_time = exponential_heat_solver_2d_boundary(nu, u0, (x0, xf), (y0, yf), tf, nx, ny, nt)

/Users/mckayladavis/opt/anaconda3/envs/acme/lib/python3.10/site-packages/scipy/sparse/_index.py:143: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray(i, j, x)
/Users/mckayladavis/opt/anaconda3/envs/acme/lib/python3.10/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:412: SparseEfficiencyWarning: splu converted its input to CSC format
  warn('splu converted its input to CSC format', SparseEfficiencyWarning)
/Users/mckayladavis/opt/anaconda3/envs/acme/lib/python3.10/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:302: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  warn('spsolve is more efficient when sparse b '
100%|██████████| 100/100 [00:00<00:00, 160.81it/s]


In [32]:
U_edt, X, Y, t_vals, end_time = exponential_heat_solver_2d_boundary(nu, u0, (x0, xf), (y0, yf), tf, nx, ny, nt, True, True)

/Users/mckayladavis/opt/anaconda3/envs/acme/lib/python3.10/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:412: SparseEfficiencyWarning: splu converted its input to CSC format
  warn('splu converted its input to CSC format', SparseEfficiencyWarning)
/Users/mckayladavis/opt/anaconda3/envs/acme/lib/python3.10/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:302: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  warn('spsolve is more efficient when sparse b '
100%|██████████| 100/100 [00:00<00:00, 155.82it/s]


In [36]:
# Create 3D plot
fig = plt.figure(figsize=(8,6))
ax1 = fig.add_subplot(111, projection='3d')

ax1.set_zlim3d([-.7,.7])

plt.suptitle(r'Solution of $u_t = u_{xx} + u_{yy}$ Including Neumann Boundary Conditions')

# Create empty line objects
# explicit = ax1.plot_surface(X, Y, U_explicit[0], cmap = 'magma',vmax = np.max(U_explicit), vmin = np.min(U_explicit))
# ax1.set_title('Explicit Method')

# crank = ax2.plot_surface(X, Y, U_crank[0], cmap = 'magma',vmax = np.max(U_crank), vmin = np.min(U_crank))
# ax2.set_title('Crank Nicolson Method')

edt = ax1.plot_surface(X, Y, U_edt[0], cmap = 'magma',vmax = np.max(U_edt), vmin = np.min(U_edt))
ax1.set_title('t = 0')

# adi = ax4.plot_surface(X, Y, U_adi[0], cmap = 'magma',vmax = np.max(U_adi), vmin = np.min(U_adi))
# ax4.set_title('ADI Method')

plt.tight_layout()

# Make update function to updata all line plots
def update(t):
    global edt
    edt.remove()
    edt = ax1.plot_surface(X, Y, U_edt[t], cmap = 'magma',vmax = np.max(U_edt), vmin = np.min(U_edt))
    ax1.set_title(f"t = {t_vals[t]}")
    
    plt.tight_layout()

    return edt


# Make figure
ani = FuncAnimation(fig, update, frames=range(len(t_vals)), interval=100)
ani.save(f'ghost_points.mp4')

plt.close()

